In [ ]:
# Verified 2026-11-02: usecase2 TESTING pipeline, execution qaq3ddxjfasy.
MODEL_ID = "usecase2"

# TESTING / GetData: raw labelled test rows.
DATA_URI = "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1102a/models/usecase2/runs/qaq3ddxjfasy/steps/GetData/raw/data.csv"
# TESTING / Preprocess: selection copied from the resolved model.
SELECTION_URI = "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1102a/models/usecase2/runs/qaq3ddxjfasy/steps/Preprocess/selection/selected_features.csv"
# TRAINING / Train: full model from 960ba26c4ae6, selected by TESTING / ResolveModel.
MODEL_URI = "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1102a/models/usecase2/runs/960ba26c4ae6/model/application/model.tar.gz"
# TESTING / Score: the entire output folder, including nested files.
SCORE_OUTPUT_URI = "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1102a/models/usecase2/runs/qaq3ddxjfasy/steps/Score/"


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import os
import sys
import tarfile

import boto3
from IPython import get_ipython

get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))

work_directory = Path(mkdtemp(prefix="postprocess-"))
data_directory = work_directory / "raw"
selection_file = work_directory / "selection/selected_features.csv"
model_archive = work_directory / "model.tar.gz"
model_directory = work_directory / "model"
prediction_directory = work_directory / "predictions"
output_directory = work_directory / "output"
output_directory.mkdir()

# Local model reads and writable output; no pipeline report upload.
os.environ.update({
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_ARTIFACT_URI": MODEL_URI,
    "MLOPS_WRITE_RUN_REPORT": "0",
})

s3 = boto3.client("s3")
for uri, destination in (
    (DATA_URI, data_directory / "data.csv"),
    (SELECTION_URI, selection_file),
    (MODEL_URI, model_archive),
):
    destination.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(uri)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(destination))

# Copy all Score outputs; keep output/ and other nested folders intact.
address = urlsplit(SCORE_OUTPUT_URI)
prefix = address.path.lstrip("/").rstrip("/") + "/"
score_files = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=address.netloc, Prefix=prefix):
    for item in page.get("Contents", []):
        key = item["Key"]
        if key.endswith("/"):
            continue
        destination = (prediction_directory / key[len(prefix):]).resolve()
        destination.relative_to(prediction_directory.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(address.netloc, key, str(destination))
        score_files += 1
print("Score files:", score_files, "|", prediction_directory)

# The entrypoint receives the extracted model directory.
with tarfile.open(model_archive) as archive:
    archive.extractall(model_directory, filter="data")

print("Inputs:", work_directory)
print("Outputs:", output_directory)


In [ ]:
from mlops_models.entrypoints import postprocess_test as postprocess
# from mlops_models.entrypoints import postprocess_inference as postprocess

# Change only the import above to switch between test and inference.
postprocess = importlib.reload(postprocess)
result_directory = postprocess.execute(
    model_id=MODEL_ID,
    input_path=prediction_directory,
    output_dir=output_directory,
    model_dir=model_directory,
    data_directory=data_directory,
    selection_file=selection_file,
)
result_directory
